# APM_50441_EP - Machine Learning 2 - Lab 5: Recommender systems and text

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC5/APM_50441_EP_Lab_5_simplified.ipynb) or run it on your own computer (see *Working environment* below).

This lab illustrates the fifth lecture (*Recommender Systems, Text and Words*). Its goals are:
- to build the baselines of a recommender system (global mean, user and item biases) and to see the long tail,
- to implement an item-based collaborative filtering,
- to complete the rating matrix with a low-rank model: SVD, Soft-Impute, alternating least squares and a matrix factorization trained by stochastic gradient descent in PyTorch,
- to evaluate the recommendations with ranking metrics, and to see that a good RMSE does not make a good top-$10$ list.

The optional *To go further* part, after the core, can be skipped at first: it covers implicit feedback, evaluation issues (time, sampled metrics, exposure bias, feedback loops), cold start and hybrid models, sequential recommendation, and the text half of the lecture (bag of words, BM25, topic models, word vectors, tokenization, RNN and attention).

We use `Python` with `numpy`, `polars`, `seaborn`, `scikit-learn` and `PyTorch`. All the models are written by hand in `numpy` or `PyTorch`: the specialized libraries (`surprise`, `implicit`, `LightFM`) are not maintained for recent versions of `numpy`.

**Working environment.** Colab is the simplest option: open the notebook of this lab from the course repository (*Open in Colab* link) and run it. To work on your own computer, download the lab folder (the notebook and its files) or clone the course repository, and open the notebook in an IDE that runs Jupyter notebooks (VS Code with the Python and Jupyter extensions, JupyterLab, Spyder, PyCharm). Create a Python environment once with [`uv`](https://docs.astral.sh/uv/): `uv venv`, activate it (`source .venv/bin/activate`, or `.venv\Scripts\activate` on Windows) and `uv pip install jupyter`, or build the whole course environment with `uv sync` in `PC/env`; then select this environment as the kernel of the notebook. The installation cell below installs the packages of the lab into the Python running the notebook (Colab or your environment), and the data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

All the libraries of the core part are installed at once.

In [ ]:
# Installs into the Python running this notebook: Colab, or locally an activated virtual environment
import sys
!uv pip install --python "{sys.executable}" "scikit-learn==1.9.1" "polars==1.44.2" "seaborn==0.13.2" "torch"

We use the MovieLens 100k data: $100\,000$ ratings ($1$ to $5$ stars) of $1682$ movies by $943$ users, collected by GroupLens in 1997-1998. Its license does not allow redistribution: it is downloaded from the GroupLens website.

In [ ]:
import os
import urllib.request
if not os.path.exists("ml-100k.zip"):
  urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-100k.zip", "ml-100k.zip")

The data and a random split of the ratings ($80\%$ training, $10\%$ validation, $10\%$ test) are given. The identifiers of the users and the movies start at $1$ in the files: they are shifted to start at $0$, to be used as indices.

In [ ]:
import io
import time
import zipfile
import numpy as np
import polars as pl
import seaborn.objects as so
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)  # small models: a few threads are faster than many

GENRES = ["unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
          "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]
with zipfile.ZipFile("ml-100k.zip") as z:
  ratings = pl.read_csv(io.BytesIO(z.read("ml-100k/u.data")), separator="\t", has_header=False,
                        new_columns=["user", "item", "rating", "timestamp"])
  movies = pl.read_csv(io.BytesIO(z.read("ml-100k/u.item")), separator="|", has_header=False, encoding="latin1",
                       new_columns=["item", "title", "release", "video release", "url"] + GENRES)
  users = pl.read_csv(io.BytesIO(z.read("ml-100k/u.user")), separator="|", has_header=False,
                      new_columns=["user", "age", "gender", "occupation", "zip"])
ratings = ratings.with_columns(pl.col("user") - 1, pl.col("item") - 1)
movies = movies.with_columns(pl.col("item") - 1)
users = users.with_columns(pl.col("user") - 1)
n_users, n_items = users.height, movies.height
titles = np.array(movies["title"].to_list())

rng = np.random.default_rng(0)
part = rng.choice(3, size=ratings.height, p=[0.8, 0.1, 0.1])
train, valid, test = [ratings.filter(pl.Series(part == k)) for k in range(3)]
def arrays(df):
  return df["user"].to_numpy(), df["item"].to_numpy(), df["rating"].to_numpy().astype(float)
u_train, i_train, r_train = arrays(train)
u_valid, i_valid, r_valid = arrays(valid)
u_test, i_test, r_test = arrays(test)
def rmse(prediction, r):
  return np.sqrt(np.mean((prediction - r) ** 2))
{'users': n_users, 'movies': n_items, 'training ratings': train.height, 'validation ratings': valid.height, 'test ratings': test.height}

## 1. The data and the baselines

**Question** What is the proportion of observed entries in the $943 \times 1682$ rating matrix? Plot the distribution of the ratings, and the number of ratings of each movie sorted in decreasing order (log scale). Which share of the ratings concerns the $10\%$ most rated movies? How many movies have fewer than $5$ ratings?

**Answer**

---

The simplest predictor is the global mean $\mu$ of the training ratings. The *bias* model (lecture: *Ratings Issues*) adds a user effect and an item effect, $\hat r_{ui} = \mu + b_u + b_i$, fitted by penalized least squares:
$$\min_{b} \sum_{(u,i) \in \Omega} (r_{ui} - \mu - b_u - b_i)^2 + \lambda\Big(\sum_u b_u^2 + \sum_i b_i^2\Big).$$
For fixed item biases, the optimal user bias is $b_u = \frac{\sum_{i \in \Omega_u}(r_{ui} - \mu - b_i)}{|\Omega_u| + \lambda}$, and symmetrically: the problem is solved by alternating these updates (`np.bincount` with weights computes the sums over users or items).

**Question** Compute the validation RMSE of the global mean. Implement the bias model, choose $\lambda \in \{1, 2, 5, 10, 25\}$ on the validation set, and compare the $5$ movies with the highest mean rating with the $5$ movies with the highest bias $b_i$ (with their numbers of ratings). Why is the penalty needed? (Store the global mean in `mu` and the biases obtained with $\lambda = 5$ in `b_user` and `b_item`: they are used below.)

**Answer**

---

## 2. Item-based collaborative filtering

Collaborative filtering predicts a rating from the ratings of similar items (or users). We work on the *residuals* of the bias model, $\tilde r_{ui} = r_{ui} - \mu - b_u - b_i$, stored in a dense matrix `R` with zeros for the missing entries and a boolean mask `W` of the observed ones. The similarity of two items is the cosine of their residual columns. As two movies seen by few common users can get a large similarity by chance, it is *shrunk* towards $0$: $s_{ij} \times \frac{n_{ij}}{n_{ij} + \beta}$, $n_{ij}$ being the number of users who rated both.

In [ ]:
R = np.zeros((n_users, n_items))
W = np.zeros((n_users, n_items), dtype=bool)
R[u_train, i_train] = r_train - mu - b_user[u_train] - b_item[i_train]
W[u_train, i_train] = True
baseline_valid = mu + b_user[u_valid] + b_item[i_valid]
baseline_test = mu + b_user[u_test] + b_item[i_test]

**Question** Compute the cosine similarities between the items, with and without the shrinkage ($\beta = 50$). Display the $5$ movies most similar to *Star Wars (1977)* in both cases. What does the shrinkage change?

**Answer**

---

**Question** Predict the residual $\tilde r_{ui}$ by the weighted mean $\sum_{j \in N_k(i, u)} s_{ij}\tilde r_{uj} / \sum_{j}|s_{ij}|$ over the $k$ items rated by $u$ that are the most similar to $i$, and add it to the bias model. Compute the validation RMSE for $k \in \{10, 30, 100\}$ with the shrunk similarity.

**Answer**

---

## 3. Matrix completion

A low-rank model assumes that the matrix of the residuals is close to $UV^\top$, with $U \in \mathbb{R}^{n \times q}$ (the user factors) and $V \in \mathbb{R}^{p \times q}$ (the item factors). The truncated SVD of the zero-filled matrix is the best rank $q$ approximation of `R`, but it treats the missing entries as observed zeros. The weighted problem only fits the observed entries (lecture: *Weighted SVD*); the iterative algorithms below solve it, or its convex relaxation with the nuclear norm (PC5, Exercise 2).

**Question** Compute the SVD of the zero-filled residual matrix (`np.linalg.svd`), and the validation RMSE of the bias model plus its rank $q$ approximation, for $q \in \{1, 2, 5, 10, 20, 50\}$.

**Answer**

---

*Hard-Impute* fills the missing entries with the current estimate $Z$ and projects on the matrices of rank $q$: $Z \leftarrow \text{SVD}_q\big(W \odot R + (1 - W) \odot Z\big)$. *Soft-Impute* replaces the projection by a soft thresholding of the singular values, $d_k \mapsto (d_k - \lambda)_+$, which is the proximal operator of $\lambda\|\cdot\|_*$: it solves $\min_Z \frac{1}{2}\|W \odot (R - Z)\|^2 + \lambda\|Z\|_*$ (PC5). A truncated randomized SVD (`randomized_svd`, $100$ components) keeps the iterations fast.

**Question** Write one function performing Hard-Impute (given a rank) or Soft-Impute (given $\lambda$), with $20$ iterations. Compute the validation RMSE of Hard-Impute for $q \in \{1, 2, 5\}$ and of Soft-Impute along the path $\lambda \in \{40, 20, 15, 10, 7\}$, each solution starting from the previous one (*warm start*); report the rank of each Soft-Impute solution.

**Answer**

---

## 4. Alternating least squares and stochastic gradient descent

The factorization can also be fitted directly on the observed entries, with a ridge penalty (lecture: *Regularized Weighted SVD*):
$$\min_{U, V}\sum_{(u,i) \in \Omega}(\tilde r_{ui} - U_u^\top V_i)^2 + \lambda\big(\|U\|^2 + \|V\|^2\big).$$
For fixed $V$, each row $U_u$ is a ridge regression on the items $\Omega_u$ rated by $u$: $U_u = (V_{\Omega_u}^\top V_{\Omega_u} + \lambda I)^{-1}V_{\Omega_u}^\top \tilde r_{u,\Omega_u}$ (PC5, Exercise 3), and symmetrically for $V$. *Alternating least squares* (ALS) alternates these exact updates.

**Question** Implement ALS ($10$ iterations, random initialization with standard deviation $0.1$), and compute its validation RMSE for $q \in \{5, 10, 20\}$ and $\lambda \in \{10, 20\}$.

**Answer**

---

Stochastic gradient descent (the *Funk SVD* of the Netflix prize) updates the factors one rating, or one mini-batch of ratings, at a time: it scales to huge datasets and to any differentiable model. In `PyTorch`, the factors and the biases are *embeddings* (tables of vectors indexed by the users and the items), and the whole model $\hat r_{ui} = \mu + b_u + b_i + U_u^\top V_i$ is trained jointly.

**Question** Write the biased matrix factorization as a `torch.nn.Module` with `nn.Embedding` layers ($q = 20$, factors initialized with standard deviation $0.01$, biases with $0$), and train it with Adam (learning rate $5\cdot10^{-3}$, mini-batches of $1024$ ratings, $30$ epochs) on the squared loss plus $\lambda$ times the squared norms of the parameters of the batch. Plot the training and validation RMSE along the epochs for $\lambda \in \{0, 0.1\}$. Which epoch would early stopping choose?

**Answer**

---

## 5. From ratings to recommendations: ranking metrics

A recommender system shows a short list of items: only the top of the list matters (lecture: *Ranking Metrics*). For each user, we rank *all* the movies he or she has not rated in the training set (ranking against a few sampled movies may change the comparison of the methods, see *Offline Evaluation Pitfalls*), keep the top $K = 10$, and consider as relevant the movies of the test set rated $4$ or $5$. The precision $P@K$ and the recall $R@K$ are the fractions of relevant movies in the list and of the relevant movies in the list; $\text{NDCG}@K$ discounts the hits by $1/\log_2(1 + \text{rank})$ and normalizes by the best possible value; the MRR is the mean inverse rank of the first hit. The *coverage* is the fraction of the catalog that appears in at least one list.

In [ ]:
relevant = {}
for u, i, r in zip(u_test, i_test, r_test):
  if r >= 4:
    relevant.setdefault(u, set()).add(i)
discount = 1 / np.log2(np.arange(2, 12))
def ranking_metrics(scores, K=10):
  """scores: n_users x n_items matrix, the higher the better; the movies of the training set are excluded."""
  scores = np.where(W, -np.inf, scores)
  top = np.argpartition(-scores, K, axis=1)[:, :K]
  top = np.take_along_axis(top, np.argsort(-np.take_along_axis(scores, top, axis=1), axis=1), axis=1)
  metrics = []
  for u, rel in relevant.items():
    hits = np.isin(top[u], list(rel))
    metrics.append({'P@10': hits.mean(), 'R@10': hits.sum() / len(rel),
                    'NDCG@10': (hits * discount).sum() / discount[:min(len(rel), K)].sum(),
                    'MRR': 1 / (np.argmax(hits) + 1) if hits.any() else 0})
  return pl.DataFrame(metrics).mean().with_columns(coverage=pl.lit(len(np.unique(top[list(relevant)])) / n_items))

**Question** *Predict before you run:* which of the methods has the best NDCG@10? Compute the test RMSE and the ranking metrics of: the popularity (number of training ratings of each movie, the same for every user), the bias model, the ALS model ($q = 10$, $\lambda = 20$) and the PyTorch factorization (trained with $\lambda = 0.1$ for the best number of epochs). Display the top $5$ of the first user for each method.

**Answer**

---

The rating models predict well how much a user would like a movie *if he or she watched it*, but the test set only contains the movies the users chose to watch and rate. A top-$N$ recommender must also predict *which* movies a user will watch: this is the *implicit feedback* view, where the data are the pairs $(u, i)$ observed, whatever the rating.

**Question** Build an implicit item-based recommender: from the binary matrix `W` (rated or not), compute the cosine similarities between the items, and score each movie for a user by the sum of its similarities with the movies he or she rated. Compare its ranking metrics with the previous ones.

**Answer**

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.